# UniDepth V2 ViTL14 — Depth Evaluation (Pre-Masked Edition)

Evaluates **UniDepth V2 ViTL14** (`lpiccinelli/unidepth-v2-vitl14`) on the
**KITTI Eigen-Split** dataset.

**Key change vs. original script:** Binary instance masks are **loaded from the
pre-generated mask store** produced by `inst_seg_notebook.ipynb` — SAM3 is
never called at runtime.

### Pipeline
| Step | What happens |
|------|-------------|
| 0 | Imports & GPU setup |
| 1 | Configuration (paths, thresholds) |
| 2 | KITTI dataset indexing |
| 3 | UniDepth V2 ViTL14 model loading |
| 4 | Pre-generated mask loading helpers |
| 5 | Depth metrics & scale helpers |
| 6 | Main processing loop → CSV |
| 7 | Analysis & 12 plots |

### Mask store layout (mirrors KITTI root)
```
MASK_ROOT / <split> / <date> / <drive> / image_02/data / <frame_stem> /
    <frame_stem>_manifest.json
    <frame_stem>_<super_cat>_<sub_cat>_<idx>_<uid8>_<score>_mask.png
```


## 0 · GPU Selection & Imports

In [1]:
import os
# os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")   # ← change GPU here

import sys
import glob
import json
import math
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F_torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D   # noqa (registers 3d projection)

import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")
print("Imports OK  |  CUDA available:", torch.cuda.is_available())


Imports OK  |  CUDA available: True


## 1 · Logging

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("unidepth_v2_eval")
logging.getLogger("transformers").setLevel(logging.ERROR)


## 2 · Configuration  *(edit paths here)*

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Using device: %s", DEVICE)

# ── Dataset ───────────────────────────────────────────────────────────────────
KITTI_ROOT      = "/STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_test"
USE_TRAIN       = False        # include training split?
MIN_DEPTH       = 1e-3        # metres
MAX_DEPTH       = 80.0        # metres
FOCAL_LENGTH_X  = 721.5577    # KITTI default; for horizontal-angle calculation

# ── UniDepth V2 ViTL14 ───────────────────────────────────────────────────────
# HuggingFace Hub (downloads on first run):
#   https://huggingface.co/lpiccinelli/unidepth-v2-vitl14
# Requires: pip install git+https://github.com/lpiccinelli-eth/UniDepth
UNIDEPTH_WEIGHTS = "/STORAGE/Arka/pretrained_weights"
CACHE_DIR        = "/STORAGE/Arka/cache"

# ── Pre-generated mask store (from inst_seg_notebook.ipynb) ──────────────────
# Mirrors KITTI_ROOT directory structure exactly:
#   MASK_ROOT / <split> / <date> / <drive> / image_02/data / <frame_stem> /
MASK_ROOT        = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output"

# ── Outputs ───────────────────────────────────────────────────────────────────
MAIN_OUTPUT_DIR  = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output/unidepth_v2_premasked_eval"
OUTPUT_CSV_PATH  = os.path.join(MAIN_OUTPUT_DIR, "unidepth_v2_results.csv")
PLOT_DIR         = os.path.join(MAIN_OUTPUT_DIR, "plots")
DEPTH_NPY_DIR    = os.path.join(MAIN_OUTPUT_DIR, "depth_maps_npy")
RGB_SAVE_DIR     = os.path.join(MAIN_OUTPUT_DIR, "rgb_images")

for _d in [MAIN_OUTPUT_DIR, PLOT_DIR, DEPTH_NPY_DIR, RGB_SAVE_DIR]:
    os.makedirs(_d, exist_ok=True)

# ── Mask confidence gate ──────────────────────────────────────────────────────
MIN_MASK_SCORE   = 0.50
MIN_VALID_PIXELS = 50

# ── Resume support ────────────────────────────────────────────────────────────
START_INDEX = 0

# ── Visualisation palette ─────────────────────────────────────────────────────
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
}
SCALE_LABELS = {
    "unscaled": "Unscaled",
    "median":   "Median Scale",
    "mean":     "Mean Scale",
    "affine":   "Affine Scale",
}

print("Configuration loaded.")


12:43:42  INFO      Using device: cuda


Configuration loaded.


## 3 · CSV Column Schema

In [4]:
CSV_COLUMNS = [
    # ── Frame & Mask Identity ────────────────────────────────────────────────
    "img_id",            # integer dataset-row index
    "file_name",         # unique string: split_drive_framestem
    "mask_id",           # local_idx from manifest (per-frame ordering)
    "mask_uid",          # globally unique UID from SAM3 manifest (uuid4 first 8 hex)
    "mask_score",        # SAM3 detection confidence
    "super_cat",         # super-category (vehicle / pedestrian / stationary_object)
    "sub_cat",           # sub-category  (vehicle_headlights, tree, …)
    "horizontal_angle",  # degrees from image principal axis

    # ── Unscaled UniDepth V2 ─────────────────────────────────────────────────
    "unscaled_rmse",      "unscaled_abs_rel",   "unscaled_delta1",
    "unscaled_avg_depth", "unscaled_min_depth",

    # ── Median Scale ─────────────────────────────────────────────────────────
    "median_scale",
    "median_rmse",        "median_abs_rel",     "median_delta1",
    "median_avg_depth",   "median_min_depth",

    # ── Mean Scale ───────────────────────────────────────────────────────────
    "mean_scale",
    "mean_rmse",          "mean_abs_rel",       "mean_delta1",
    "mean_avg_depth",     "mean_min_depth",

    # ── Affine Scale ─────────────────────────────────────────────────────────
    "affine_scale",       "affine_shift",
    "affine_rmse",        "affine_abs_rel",     "affine_delta1",
    "affine_avg_depth",   "affine_min_depth",

    # ── LiDAR Ground-Truth Reference ─────────────────────────────────────────
    "lidar_avg_depth",    "lidar_min_depth",

    # ── File Paths ────────────────────────────────────────────────────────────
    "rgb_image_path",     "gt_depth_map_path",
    "unidepth_map_path",  "mask_path",
]

print(f"CSV_COLUMNS defined ({len(CSV_COLUMNS)} columns).")


CSV_COLUMNS defined (38 columns).


## 4 · KITTI Dataset Indexing

In [5]:
def index_kitti(kitti_root: str = KITTI_ROOT,
                use_train: bool = USE_TRAIN) -> pd.DataFrame:
    """
    Walk the KITTI Eigen-Split directory and return a DataFrame of
    (folder, sequence, drive, filename, gt_path, rgb_path) pairs.

    Expected layout under kitti_root:
        <split> / <date> / <drive>_sync / proj_depth / groundtruth / image_02 / <frame>.png
        <split> / <date> / <drive>_sync / image_02 / data / <frame>.png
    """
    rows = []

    def _collect(split_root: str, tag: str) -> None:
        pattern  = os.path.join(
            split_root, "*", "*_sync",
            "proj_depth", "groundtruth", "image_02", "*.png"
        )
        gt_paths = glob.glob(pattern)
        for gt_path in tqdm(gt_paths, desc=f"Indexing {tag}", leave=False):
            parts    = gt_path.split(os.sep)
            date     = parts[-6]
            drive    = parts[-5]
            fname    = parts[-1]
            rgb_path = os.path.join(split_root, date, drive,
                                    "image_02", "data", fname)
            if not os.path.exists(rgb_path):
                continue
            rows.append(dict(
                folder=tag, sequence=date, drive=drive,
                filename=fname, gt_path=gt_path, rgb_path=rgb_path,
            ))

    _collect(os.path.join(kitti_root, "test"),  "Test")
    if use_train:
        _collect(os.path.join(kitti_root, "train"), "Train")

    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Dataset indexed: %d frames, %d drives.",
             len(df), df["drive"].nunique() if len(df) else 0)
    return df

print("index_kitti() defined.")


index_kitti() defined.


## 5 · Helpers: Eigen Crop & Depth Display

In [6]:
def eigen_crop_mask(shape: tuple) -> np.ndarray:
    """Boolean mask for the standard Eigen / Monodepth2 evaluation crop."""
    h, w = shape[:2]
    m    = np.zeros((h, w), dtype=bool)
    top  = 44
    bot  = min(top + 352, h)
    m[top:bot, 0:w] = True
    return m


def normalize_depth_for_display(depth: np.ndarray,
                                  cmap=plt.cm.magma) -> np.ndarray:
    """Convert a float depth map to H×W×4 uint8 RGBA for display."""
    valid = depth > 0
    if not np.any(valid):
        return np.zeros((*depth.shape, 4), dtype=np.uint8)
    vmin = float(np.percentile(depth[valid], 2))
    vmax = float(np.percentile(depth[valid], 98))
    norm = np.clip((depth - vmin) / max(vmax - vmin, 1e-8), 0.0, 1.0)
    return (cmap(norm) * 255).astype(np.uint8)

print("Crop & display helpers defined.")


Crop & display helpers defined.


## 6 · Pre-Generated Mask Loading  *(replaces SAM3 inference)*

The two functions below resolve the mask directory for any KITTI frame and
load all pre-generated instance masks from it.  The manifest JSON is used
when available (fast path); otherwise file names are parsed directly.


In [7]:
def get_mask_dir(rgb_path: str,
                 kitti_root: str = KITTI_ROOT,
                 mask_root: str  = MASK_ROOT) -> str:
    """
    Derive the pre-generated mask directory for a given KITTI RGB image path.

    Mapping (mirrors inst_seg_notebook.ipynb output layout):
        KITTI_ROOT / <split> / <date> / <drive> / image_02/data / <frame>.png
           →
        MASK_ROOT  / <split> / <date> / <drive> / image_02/data / <frame_stem> /
    """
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), kitti_root)
    return os.path.join(mask_root, rel_data_dir, frame_stem)


def load_masks_for_frame(mask_dir: str,
                          min_score: float = MIN_MASK_SCORE) -> list:
    """
    Load all pre-generated instance masks for one frame.

    Prefers the JSON manifest (fast path).  Falls back to glob + filename
    parsing when the manifest is absent.

    Filename format (inst_seg_notebook.ipynb convention):
        <frame_stem>_<super_cat>_<sub_cat>_<local_idx:04d>_<uid8>_<score:.3f>_mask.png

    Returns list of dicts:
        uid, local_idx, super_cat, sub_cat, score, box,
        mask (H×W bool ndarray), path (Path)
    """
    mask_dir_path = Path(mask_dir)
    if not mask_dir_path.exists():
        return []

    frame_stem    = mask_dir_path.name
    manifest_path = mask_dir_path / f"{frame_stem}_manifest.json"

    # ── Fast path: JSON manifest ──────────────────────────────────────────────
    if manifest_path.exists():
        with open(manifest_path, "r") as fh:
            records = json.load(fh)
        instances = []
        for rec in records:
            if float(rec["score"]) < min_score:
                continue
            mask_file = mask_dir_path / rec["mask_file"]
            if not mask_file.exists():
                log.warning("Mask file missing: %s", mask_file)
                continue
            mask_np = np.array(Image.open(mask_file).convert("L")) > 127
            instances.append({
                "uid":       rec["uid"],
                "local_idx": int(rec["local_idx"]),
                "super_cat": rec["super_cat"],
                "sub_cat":   rec["sub_cat"],
                "score":     float(rec["score"]),
                "box":       rec.get("box"),
                "mask":      mask_np,
                "path":      mask_file,
            })
        return instances

    # ── Fallback: glob + filename parsing ─────────────────────────────────────
    instances = []
    for mask_file in sorted(mask_dir_path.glob("*_mask.png")):
        stem = mask_file.stem[:-5]                   # strip "_mask"
        if not stem.startswith(frame_stem + "_"):
            continue
        suffix = stem[len(frame_stem) + 1:]
        parts = suffix.rsplit("_", 3)                # [cat_str, local_idx, uid, score]
        if len(parts) != 4:
            log.warning("Unexpected mask filename: %s", mask_file.name)
            continue
        cat_str, idx_str, uid_str, score_str = parts
        cat_parts = cat_str.split("_", 1)
        if len(cat_parts) != 2:
            continue
        super_cat, sub_cat = cat_parts
        try:
            local_idx = int(idx_str)
            score     = float(score_str)
        except ValueError:
            continue
        if score < min_score:
            continue
        mask_np = np.array(Image.open(mask_file).convert("L")) > 127
        instances.append({
            "uid":       uid_str,
            "local_idx": local_idx,
            "super_cat": super_cat,
            "sub_cat":   sub_cat,
            "score":     score,
            "box":       None,
            "mask":      mask_np,
            "path":      mask_file,
        })

    instances.sort(key=lambda x: x["local_idx"])
    return instances

print("Mask loading helpers defined.")


Mask loading helpers defined.


## 7 · UniDepth V2 ViTL14 Model Loading & Inference

**HuggingFace:** `lpiccinelli/unidepth-v2-vitl14`  
**GitHub:** https://github.com/lpiccinelli-eth/UniDepth  
**Install:** `pip install git+https://github.com/lpiccinelli-eth/UniDepth`


In [8]:
def load_unidepth_v2(weights_path: str = UNIDEPTH_WEIGHTS,
                     cache_dir: str    = CACHE_DIR,
                     device: torch.device = DEVICE):
    """Load UniDepth V2 ViTL14 from HuggingFace Hub or a local directory."""
    from unidepth.models import UniDepthV2

    log.info("Loading UniDepth V2 ViTL14 from '%s' …", weights_path)
    model = (
        UniDepthV2
        .from_pretrained(weights_path, cache_dir=cache_dir)
        .to(device)
        .eval()
    )
    log.info("UniDepth V2 ViTL14 ready on %s.", device)
    return model


@torch.no_grad()
def predict_unidepth(model, image_np: np.ndarray,
                     device: torch.device = DEVICE) -> np.ndarray:
    """
    Run UniDepth V2 ViTL14 inference on a single RGB image.

    Parameters
    ----------
    image_np : H×W×3 uint8 numpy array (RGB).

    Returns
    -------
    depth : H×W float32 metric depth map in metres.
    """
    rgb   = torch.from_numpy(image_np).permute(2, 0, 1).to(device)  # C H W
    preds = model.infer(rgb)
    return preds["depth"].squeeze().cpu().numpy().astype(np.float32)

print("UniDepth V2 ViTL14 helpers defined.")


UniDepth V2 ViTL14 helpers defined.


## 8 · Depth Metrics & Scale Computation

In [9]:
def compute_depth_errors(gt: np.ndarray,
                         pred: np.ndarray,
                         mask: np.ndarray) -> dict:
    """Standard KITTI / Monodepth2 depth metrics on masked pixels only."""
    gt_v   = gt[mask]
    pred_v = np.maximum(pred[mask], 1e-8)

    thresh   = np.maximum(gt_v / pred_v, pred_v / gt_v)
    rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2)))
    abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v))
    sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / gt_v))
    rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2)))
    delta1   = float(np.mean(thresh < 1.25))
    delta2   = float(np.mean(thresh < 1.25 ** 2))
    delta3   = float(np.mean(thresh < 1.25 ** 3))
    return dict(rmse=rmse, abs_rel=abs_rel, sq_rel=sq_rel,
                rmse_log=rmse_log, delta1=delta1, delta2=delta2, delta3=delta3)


def compute_all_scales(gt_depth: np.ndarray,
                       pred_raw: np.ndarray,
                       mask: np.ndarray) -> dict:
    """
    Compute four scale-aligned depth maps from the same raw prediction.

    Scale methods
    ─────────────
    unscaled : no correction
    median   : scale = median(GT) / median(pred)   (ratio of medians)
    mean     : scale = mean(GT)   / mean(pred)     (ratio of means)
    affine   : OLS  GT ≈ a·pred + b  (scale=a, shift=b)

    Returns dict keyed by method name; each value contains:
        scale, shift, depth_map,
        rmse, abs_rel, sq_rel, rmse_log, delta1, delta2, delta3,
        avg_depth, min_depth
    """
    gt_v   = gt_depth[mask]
    pred_c = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)

    results = {}

    # ── Unscaled ──────────────────────────────────────────────────────────────
    m0 = compute_depth_errors(gt_depth, pred_c, mask)
    results["unscaled"] = {
        "scale": 1.0, "shift": 0.0, "depth_map": pred_c, **m0,
        "avg_depth": float(np.mean(pred_c[mask])),
        "min_depth": float(np.min(pred_c[mask])),
    }

    # ── Median Scale ──────────────────────────────────────────────────────────
    scale_med = float(np.median(gt_v) / (np.median(pred_c[mask]) + 1e-8))
    pred_med  = np.clip(pred_c * scale_med, MIN_DEPTH, MAX_DEPTH)
    m1 = compute_depth_errors(gt_depth, pred_med, mask)
    results["median"] = {
        "scale": scale_med, "shift": 0.0, "depth_map": pred_med, **m1,
        "avg_depth": float(np.mean(pred_med[mask])),
        "min_depth": float(np.min(pred_med[mask])),
    }

    # ── Mean Scale ────────────────────────────────────────────────────────────
    scale_mean = float(np.mean(gt_v) / (np.mean(pred_c[mask]) + 1e-8))
    pred_mean  = np.clip(pred_c * scale_mean, MIN_DEPTH, MAX_DEPTH)
    m2 = compute_depth_errors(gt_depth, pred_mean, mask)
    results["mean"] = {
        "scale": scale_mean, "shift": 0.0, "depth_map": pred_mean, **m2,
        "avg_depth": float(np.mean(pred_mean[mask])),
        "min_depth": float(np.min(pred_mean[mask])),
    }

    # ── Affine Scale (OLS) ────────────────────────────────────────────────────
    pred_flat       = pred_c[mask].ravel()
    A               = np.vstack([pred_flat, np.ones_like(pred_flat)]).T
    params, _, _, _ = np.linalg.lstsq(A, gt_v.ravel(), rcond=None)
    scale_aff       = float(params[0])
    shift_aff       = float(params[1])
    pred_aff        = np.clip(scale_aff * pred_c + shift_aff, MIN_DEPTH, MAX_DEPTH)
    m3 = compute_depth_errors(gt_depth, pred_aff, mask)
    results["affine"] = {
        "scale": scale_aff, "shift": shift_aff, "depth_map": pred_aff, **m3,
        "avg_depth": float(np.mean(pred_aff[mask])),
        "min_depth": float(np.min(pred_aff[mask])),
    }

    return results


def horizontal_angle(mask: np.ndarray, img_width: int,
                     focal_x: float = FOCAL_LENGTH_X) -> float:
    """Horizontal angle of the mask centroid relative to principal axis (°)."""
    cols = np.where(mask)[1]
    if len(cols) == 0:
        return float("nan")
    centroid_x = float(cols.mean())
    return math.degrees(math.atan((centroid_x - img_width / 2.0) / (focal_x + 1e-8)))

print("Metric & scale helpers defined.")


Metric & scale helpers defined.


## 9 · Main Processing Loop

In [10]:
def run_evaluation(
    kitti_root:     str   = KITTI_ROOT,
    mask_root:      str   = MASK_ROOT,
    weights_path:   str   = UNIDEPTH_WEIGHTS,
    cache_dir:      str   = CACHE_DIR,
    output_csv:     str   = OUTPUT_CSV_PATH,
    depth_npy_dir:  str   = DEPTH_NPY_DIR,
    rgb_save_dir:   str   = RGB_SAVE_DIR,
    use_train:      bool  = USE_TRAIN,
    min_depth:      float = MIN_DEPTH,
    max_depth:      float = MAX_DEPTH,
    min_mask_score: float = MIN_MASK_SCORE,
    min_valid_px:   int   = MIN_VALID_PIXELS,
    start_index:    int   = START_INDEX,
) -> pd.DataFrame:
    """
    Full evaluation pipeline.
      1. Resolve pre-generated mask directory for each frame.
      2. Load SAM3 masks from disk (skips frame if none found).
      3. Run UniDepth V2 ViTL14 inference (only when masks exist).
      4. For each mask compute all four scale variants + full metrics.
      5. Append one row per mask to OUTPUT_CSV_PATH.
    Returns the full results DataFrame.
    """
    dataset_df = index_kitti(kitti_root, use_train)
    if dataset_df.empty:
        log.error("No image pairs found — check KITTI_ROOT: %s", kitti_root)
        return pd.DataFrame(columns=CSV_COLUMNS)

    model = load_unidepth_v2(weights_path, cache_dir, DEVICE)

    if not os.path.exists(output_csv):
        pd.DataFrame(columns=CSV_COLUMNS).to_csv(output_csv, index=False)

    frames_no_masks = 0

    for index, row in tqdm(dataset_df.iterrows(),
                           total=len(dataset_df), desc="UniDepth V2 Eval"):
        if index < start_index:
            continue

        folder     = row["folder"]
        drive      = row["drive"]
        fname      = row["filename"]
        rgb_path   = row["rgb_path"]
        gt_path    = row["gt_path"]
        frame_stem = os.path.splitext(fname)[0]
        base_fn    = f"{folder}_{drive}_{frame_stem}"

        try:
            # ── Pre-generated masks ───────────────────────────────────────────
            mask_dir      = get_mask_dir(rgb_path, kitti_root, mask_root)
            all_instances = load_masks_for_frame(mask_dir, min_score=min_mask_score)

            if not all_instances:
                frames_no_masks += 1
                log.debug("No masks for %s (dir: %s)", base_fn, mask_dir)
                continue

            # ── Load RGB + LiDAR GT ───────────────────────────────────────────
            image_np = np.array(Image.open(rgb_path).convert("RGB"))
            gt_depth = np.array(Image.open(gt_path)).astype(np.uint16) / 256.0
            H, W     = gt_depth.shape

            # ── UniDepth V2 ViTL14 inference ──────────────────────────────────
            pred_depth = predict_unidepth(model, image_np, DEVICE)

            # Bilinear upsample to GT resolution if shapes differ
            if pred_depth.shape != (H, W):
                pred_t     = torch.from_numpy(pred_depth)[None, None]
                pred_t     = F_torch.interpolate(pred_t, size=(H, W),
                                                  mode="bilinear",
                                                  align_corners=False)
                pred_depth = pred_t.squeeze().numpy().astype(np.float32)

            crop_mask = eigen_crop_mask(gt_depth.shape)

            # ── Save RGB (once per frame) ─────────────────────────────────────
            rgb_out = os.path.join(rgb_save_dir, f"{base_fn}.png")
            if not os.path.exists(rgb_out):
                Image.fromarray(image_np).save(rgb_out)

            # ── Save depth maps (once per frame) ──────────────────────────────
            pred_npy = os.path.join(depth_npy_dir, f"{base_fn}_unidepth.npy")
            if not os.path.exists(pred_npy):
                np.save(pred_npy, pred_depth)
            gt_npy = os.path.join(depth_npy_dir, f"{base_fn}_gt.npy")
            if not os.path.exists(gt_npy):
                np.save(gt_npy, gt_depth)

            # ── Per-mask evaluation ───────────────────────────────────────────
            batch_rows = []
            for inst in all_instances:
                seg_mask  = inst["mask"]
                score     = inst["score"]
                super_cat = inst["super_cat"]
                sub_cat   = inst["sub_cat"]
                mask_idx  = inst["local_idx"]
                mask_uid  = inst.get("uid", "")

                if seg_mask.shape != (H, W):
                    log.warning(
                        "Mask shape %s != image shape %s for %s uid=%s; skipping.",
                        seg_mask.shape, (H, W), base_fn, mask_uid,
                    )
                    continue

                final_mask = (seg_mask
                              & (gt_depth > min_depth)
                              & (gt_depth < max_depth)
                              & crop_mask)

                if np.sum(final_mask) < min_valid_px:
                    continue

                sr = compute_all_scales(gt_depth, pred_depth, final_mask)

                lidar_vals = gt_depth[final_mask]
                lidar_avg  = float(np.mean(lidar_vals))
                lidar_min  = float(np.min(lidar_vals))
                h_angle    = horizontal_angle(seg_mask, W)

                batch_rows.append({
                    "img_id":             index,
                    "file_name":          base_fn,
                    "mask_id":            mask_idx,
                    "mask_uid":           mask_uid,
                    "mask_score":         score,
                    "super_cat":          super_cat,
                    "sub_cat":            sub_cat,
                    "horizontal_angle":   h_angle,
                    "unscaled_rmse":      sr["unscaled"]["rmse"],
                    "unscaled_abs_rel":   sr["unscaled"]["abs_rel"],
                    "unscaled_delta1":    sr["unscaled"]["delta1"],
                    "unscaled_avg_depth": sr["unscaled"]["avg_depth"],
                    "unscaled_min_depth": sr["unscaled"]["min_depth"],
                    "median_scale":       sr["median"]["scale"],
                    "median_rmse":        sr["median"]["rmse"],
                    "median_abs_rel":     sr["median"]["abs_rel"],
                    "median_delta1":      sr["median"]["delta1"],
                    "median_avg_depth":   sr["median"]["avg_depth"],
                    "median_min_depth":   sr["median"]["min_depth"],
                    "mean_scale":         sr["mean"]["scale"],
                    "mean_rmse":          sr["mean"]["rmse"],
                    "mean_abs_rel":       sr["mean"]["abs_rel"],
                    "mean_delta1":        sr["mean"]["delta1"],
                    "mean_avg_depth":     sr["mean"]["avg_depth"],
                    "mean_min_depth":     sr["mean"]["min_depth"],
                    "affine_scale":       sr["affine"]["scale"],
                    "affine_shift":       sr["affine"]["shift"],
                    "affine_rmse":        sr["affine"]["rmse"],
                    "affine_abs_rel":     sr["affine"]["abs_rel"],
                    "affine_delta1":      sr["affine"]["delta1"],
                    "affine_avg_depth":   sr["affine"]["avg_depth"],
                    "affine_min_depth":   sr["affine"]["min_depth"],
                    "lidar_avg_depth":    lidar_avg,
                    "lidar_min_depth":    lidar_min,
                    "rgb_image_path":     rgb_out,
                    "gt_depth_map_path":  gt_npy,
                    "unidepth_map_path":  pred_npy,
                    "mask_path":          str(inst["path"]),
                })

            if not batch_rows:
                continue

            pd.DataFrame(batch_rows, columns=CSV_COLUMNS).to_csv(
                output_csv, mode="a", header=False, index=False
            )

        except Exception as exc:
            log.error("[%s] ERROR: %s", base_fn, exc, exc_info=True)
            continue

    if frames_no_masks > 0:
        log.warning("%d frames had no pre-generated masks in MASK_ROOT.",
                    frames_no_masks)
    log.info("Evaluation complete → %s", output_csv)
    return pd.read_csv(output_csv)

print("run_evaluation() defined.")


run_evaluation() defined.


## 10 · Analysis & Visualisation Helpers

In [11]:
def load_results(csv_path: str = OUTPUT_CSV_PATH) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    return df.dropna(subset=[
        "unscaled_rmse", "median_rmse", "mean_rmse", "affine_rmse",
        "lidar_avg_depth", "lidar_min_depth",
    ])


def _save_fig(fig: plt.Figure, name: str, dpi: int = 150) -> None:
    os.makedirs(PLOT_DIR, exist_ok=True)
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    log.info("  ✓  %s", name)

print("Analysis helpers defined.")


Analysis helpers defined.


## 11 · Plot Functions (12 plots)

In [12]:
# ── Plot 00: Summary Table ────────────────────────────────────────────────────
def plot_summary_table(df: pd.DataFrame) -> None:
    specs = [
        ("Unscaled",     "unscaled_rmse","unscaled_abs_rel","unscaled_delta1",None,None),
        ("Median Scale", "median_rmse",  "median_abs_rel",  "median_delta1",  "median_scale",None),
        ("Mean Scale",   "mean_rmse",    "mean_abs_rel",    "mean_delta1",    "mean_scale",  None),
        ("Affine Scale", "affine_rmse",  "affine_abs_rel",  "affine_delta1",  "affine_scale","affine_shift"),
    ]
    rows_data = []
    for label, rc, ac, dc, sc, shc in specs:
        sv  = df[sc].mean()  if sc  else 1.0
        shv = df[shc].mean() if shc else 0.0
        rows_data.append([
            label, f"{sv:.4f}", f"{shv:.4f}" if shc else "—",
            f"{df[rc].mean():.4f}", f"{df[ac].mean():.4f}", f"{df[dc].mean():.4f}",
        ])
    fig, ax = plt.subplots(figsize=(14, 3)); ax.axis("off")
    tbl = ax.table(cellText=rows_data,
                   colLabels=["Scale Method","Avg Scale","Avg Shift",
                               "Avg RMSE (m)","Avg Abs Rel","Avg δ<1.25"],
                   cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1.2, 1.9)
    for j in range(6):
        tbl[0, j].set_facecolor("#2c3e50"); tbl[0, j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows_data) + 1):
        bg = "white" if i % 2 == 0 else "#f0f4f8"
        for j in range(6): tbl[i, j].set_facecolor(bg)
    fig.suptitle("UniDepth V2 ViTL14 — Scale Method Summary", fontsize=14, fontweight="bold", y=1.04)
    plt.tight_layout(); _save_fig(fig, "00_summary_table.png", dpi=180)


# ── Plot 01: GT vs Predicted Depth ────────────────────────────────────────────
def plot_gt_vs_pred(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_avg_depth"),("Median Scale","median_avg_depth"),
                ("Mean Scale","mean_avg_depth"),("Affine Scale","affine_avg_depth")]
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sub = df[["lidar_avg_depth", ycol]].dropna()
        sns.regplot(ax=ax, data=sub, x="lidar_avg_depth", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_title(f"UniDepth V2 ViTL14  |  {title}", fontsize=12)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH); ax.legend(fontsize=9)
    fig.suptitle("Ground-Truth vs Predicted Depth — all scale variants", fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "01_gt_vs_pred_all_scales.png")


# ── Plot 02: GT Depth vs RMSE ─────────────────────────────────────────────────
def plot_gt_vs_rmse(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    ymax = df[[v[1] for v in variants]].quantile(0.99).max() * 1.12
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sub = df[["lidar_avg_depth", ycol]].dropna()
        sns.regplot(ax=ax, data=sub, x="lidar_avg_depth", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"UniDepth V2 ViTL14  |  {title}", fontsize=12)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("RMSE (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, ymax)
    fig.suptitle("GT Depth vs RMSE — all scale variants", fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "02_gt_vs_rmse_all_scales.png")


# ── Plot 03: Horizontal Angle vs RMSE ────────────────────────────────────────
def plot_angle_vs_rmse(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    ymax = df[[v[1] for v in variants]].quantile(0.99).max() * 1.12
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sub = df[["horizontal_angle", ycol]].dropna()
        sns.regplot(ax=ax, data=sub, x="horizontal_angle", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"UniDepth V2 ViTL14  |  {title}: Angle vs RMSE", fontsize=12)
        ax.set_xlabel("Horizontal Angle (°)"); ax.set_ylabel("RMSE (m)"); ax.set_ylim(0, ymax)
    fig.suptitle("Horizontal Angle vs RMSE — all scale variants", fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "03_angle_vs_rmse_all_scales.png")


# ── Plot 04: Scale Method Comparison Bars ─────────────────────────────────────
def plot_scale_comparison_bars(df: pd.DataFrame) -> None:
    metrics_spec = [
        ("RMSE (m)  ↓", ["unscaled_rmse",   "median_rmse",   "mean_rmse",   "affine_rmse"]),
        ("Abs Rel   ↓", ["unscaled_abs_rel", "median_abs_rel","mean_abs_rel","affine_abs_rel"]),
        ("δ < 1.25  ↑", ["unscaled_delta1",  "median_delta1", "mean_delta1", "affine_delta1"]),
    ]
    labels = ["Unscaled","Median","Mean","Affine"]
    colors = list(SCALE_COLORS.values())
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, (metric_name, cols) in zip(axes, metrics_spec):
        vals = [df[c].mean() for c in cols]
        bars = ax.bar(labels, vals, color=colors, edgecolor="white", width=0.6)
        ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=10)
        ax.set_title(metric_name, fontsize=13); ax.set_ylabel(metric_name)
        ax.set_ylim(0, max(vals) * 1.22); ax.grid(axis="y", alpha=0.35)
    fig.suptitle("UniDepth V2 ViTL14 — Scale Method Comparison", fontsize=15, y=1.02)
    plt.tight_layout(); _save_fig(fig, "04_scale_comparison_bars.png")


# ── Plot 05: Model vs LiDAR by Object Category ───────────────────────────────
def plot_model_vs_lidar_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique()); n = len(cats)
    if n == 0: return
    fig, axes = plt.subplots(1, max(n, 1), figsize=(7 * n, 6))
    if n == 1: axes = [axes]
    for ax, cat in zip(axes, cats):
        sub = df[df["super_cat"] == cat]
        for key in ["unscaled","median","mean","affine"]:
            ax.scatter(sub["lidar_avg_depth"], sub[f"{key}_avg_depth"],
                       s=18, alpha=0.4, c=SCALE_COLORS[key], label=SCALE_LABELS[key])
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_title(f"Category: {cat}", fontsize=12); ax.legend(fontsize=8)
    fig.suptitle("Predicted Depth vs LiDAR GT — by Object Category", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "05_model_vs_lidar_by_category.png")


# ── Plot 06: RMSE Distribution Histograms ─────────────────────────────────────
def plot_rmse_distributions(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    fig, axes = plt.subplots(2, 2, figsize=(16, 10)); axes = axes.flatten()
    for ax, (label, col), color in zip(axes, variants, SCALE_COLORS.values()):
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.72, edgecolor="white")
        ax.axvline(vals.mean(), color="black", linestyle="--", linewidth=1.8,
                   label=f"Mean   = {vals.mean():.3f} m")
        ax.axvline(vals.median(), color="grey", linestyle=":", linewidth=1.8,
                   label=f"Median = {vals.median():.3f} m")
        ax.set_title(f"RMSE Distribution — {label}", fontsize=12)
        ax.set_xlabel("RMSE (m)"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle("RMSE Distributions Across All Masks", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "06_rmse_distributions.png")


# ── Plot 07: Scale Factor Distributions ──────────────────────────────────────
def plot_scale_distributions(df: pd.DataFrame) -> None:
    spec = [("median_scale","Median Scale",SCALE_COLORS["median"]),
            ("mean_scale","Mean Scale",SCALE_COLORS["mean"]),
            ("affine_scale","Affine Scale",SCALE_COLORS["affine"])]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.72, edgecolor="white")
        ax.axvline(1.0, color="black", linestyle="--", linewidth=1.8, label="Scale = 1.0")
        ax.axvline(vals.mean(), color="red", linestyle=":", linewidth=1.8,
                   label=f"Mean = {vals.mean():.3f}")
        ax.set_title(f"{label} Distribution", fontsize=12)
        ax.set_xlabel("Scale Factor"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle("Scale Factor Distributions — UniDepth V2 ViTL14", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "07_scale_distributions.png")


# ── Plot 08: 3D Error Surfaces ────────────────────────────────────────────────
def plot_3d_error_surfaces(df: pd.DataFrame, degree: int = 2) -> None:
    variants = [("Unscaled","unscaled_rmse","b"),("Median Scale","median_rmse","g"),
                ("Mean Scale","mean_rmse","orange"),("Affine Scale","affine_rmse","purple")]
    xcol, ycol = "lidar_avg_depth", "horizontal_angle"
    df_c = df.dropna(subset=[xcol, ycol] + [v[1] for v in variants])
    x_min, x_max = df_c[xcol].min(), df_c[xcol].max()
    y_min, y_max = df_c[ycol].min(), df_c[ycol].max()
    z_min = df_c[[v[1] for v in variants]].min().min()
    z_max = df_c[[v[1] for v in variants]].max().max()
    buf = lambda r: r * 0.08
    fig = plt.figure(figsize=(26, 6))
    fig.suptitle(f"UniDepth V2 ViTL14 — 3D Error Analysis (poly deg={degree})", fontsize=14)
    for sp_idx, (label, zcol, color) in enumerate(variants, start=1):
        ax = fig.add_subplot(1, 4, sp_idx, projection="3d")
        ax.scatter(df_c[xcol], df_c[ycol], df_c[zcol], c=color, alpha=0.35, s=8)
        Xf = df_c[[xcol, ycol]].values; Zf = df_c[zcol].values
        pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        pipe.fit(Xf, Zf)
        xs, ys = np.meshgrid(np.linspace(x_min, x_max, 40), np.linspace(y_min, y_max, 40))
        zs = pipe.predict(np.c_[xs.ravel(), ys.ravel()]).reshape(xs.shape)
        ax.plot_surface(xs, ys, zs, color=color, alpha=0.22, rstride=1, cstride=1)
        bx = buf(x_max-x_min); by = buf(y_max-y_min); bz = buf(z_max-z_min)
        ax.set_xlim(x_min-bx, x_max+bx); ax.set_ylim(y_min-by, y_max+by); ax.set_zlim(z_min-bz, z_max+bz)
        ax.set_xlabel("GT Depth (m)", fontsize=8); ax.set_ylabel("H. Angle (°)", fontsize=8)
        ax.set_zlabel("RMSE (m)", fontsize=8); ax.set_title(label, fontsize=11)
    plt.tight_layout(); _save_fig(fig, "08_3d_error_surfaces.png")


# ── Plot 09: Depth-Bin Bias Analysis ─────────────────────────────────────────
def plot_depth_bin_analysis(df: pd.DataFrame) -> None:
    bins   = np.arange(0, MAX_DEPTH + 10, 10)
    labels = [f"{int(b)}–{int(b+10)}m" for b in bins[:-1]]
    df2    = df.copy()
    df2["depth_bin"] = pd.cut(df2["lidar_avg_depth"], bins=bins, labels=labels)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    ax = axes[0]
    for key in ["unscaled","median","mean","affine"]:
        agg = df2.groupby("depth_bin", observed=True)[f"{key}_rmse"].mean()
        ax.plot(agg.index, agg.values, "o-", color=SCALE_COLORS[key],
                label=SCALE_LABELS[key], linewidth=2)
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE by Depth Bin — Scale Comparison")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    ax = axes[1]
    for key in ["unscaled","median","mean","affine"]:
        agg_pred  = df2.groupby("depth_bin", observed=True)[f"{key}_avg_depth"].mean()
        agg_lidar = df2.groupby("depth_bin", observed=True)["lidar_avg_depth"].mean()
        bias = agg_pred - agg_lidar
        ax.plot(bias.index, bias.values, "o-", color=SCALE_COLORS[key],
                label=SCALE_LABELS[key], linewidth=2)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.5, label="Zero bias")
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Prediction Bias (m)")
    ax.set_title("Depth Bias by Bin  (Pred − LiDAR)")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    fig.suptitle("Depth-Bin Analysis — UniDepth V2 ViTL14", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "09_depth_bin_analysis.png")


# ── Plot 10: RMSE per Object Category ────────────────────────────────────────
def plot_rmse_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    if len(cats) < 2: return
    keys, n_keys = ["unscaled","median","mean","affine"], 4
    x = np.arange(len(cats)); w = 0.18
    fig, ax = plt.subplots(figsize=(13, 6))
    for i, key in enumerate(keys):
        vals   = [df[df["super_cat"] == c][f"{key}_rmse"].mean() for c in cats]
        offset = (i - (n_keys - 1) / 2.0) * w
        bars   = ax.bar(x + offset, vals, width=w,
                        label=SCALE_LABELS[key], color=SCALE_COLORS[key])
        ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
    ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=11)
    ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE per Object Category — UniDepth V2 ViTL14", fontsize=13)
    ax.legend(); ax.grid(axis="y", alpha=0.35)
    plt.tight_layout(); _save_fig(fig, "10_rmse_by_category.png")


# ── Plot 11: Scale Factor vs GT Depth ────────────────────────────────────────
def plot_scale_vs_depth(df: pd.DataFrame) -> None:
    spec = [("median_scale","Median Scale",SCALE_COLORS["median"]),
            ("mean_scale","Mean Scale",SCALE_COLORS["mean"]),
            ("affine_scale","Affine Scale",SCALE_COLORS["affine"])]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        sub = df[["lidar_avg_depth", col]].dropna()
        sns.regplot(ax=ax, data=sub, x="lidar_avg_depth", y=col,
                    scatter_kws={"s": 18, "alpha": 0.30, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.axhline(1.0, color="black", linestyle="--", linewidth=1.5, label="Scale = 1.0")
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Scale Factor")
        ax.set_title(f"{label} vs Depth"); ax.legend(fontsize=9)
    fig.suptitle("Scale Factor vs Ground-Truth Depth", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "11_scale_vs_depth.png")


# ── Master caller ─────────────────────────────────────────────────────────────
def generate_all_plots(df: pd.DataFrame) -> None:
    sns.set_style("whitegrid"); sns.set_context("paper", font_scale=1.1)
    log.info("Generating analysis plots → %s", PLOT_DIR)
    plot_summary_table(df);         plot_gt_vs_pred(df)
    plot_gt_vs_rmse(df);            plot_angle_vs_rmse(df)
    plot_scale_comparison_bars(df); plot_model_vs_lidar_by_category(df)
    plot_rmse_distributions(df);    plot_scale_distributions(df)
    plot_3d_error_surfaces(df);     plot_depth_bin_analysis(df)
    plot_rmse_by_category(df);      plot_scale_vs_depth(df)
    log.info("All 12 plots saved to %s", PLOT_DIR)

print("All plot functions defined.")


All plot functions defined.


## 12 · Terminal Summary Printer

In [13]:
def print_summary(df: pd.DataFrame) -> None:
    W = 110
    print("\n" + "═" * W)
    print("  UniDepth V2 ViTL14 — Evaluation Summary  (Pre-Masked Edition)")
    print(f"  Masks evaluated: {len(df):,}   |   Unique frames: {df['img_id'].nunique():,}")
    print("═" * W)
    hdr = (f"{'Scale Method':<18}"
           f"{'Avg Scale':>11}{'Avg Shift':>11}"
           f"{'RMSE (m)':>11}{'Abs Rel':>11}"
           f"{'δ<1.25':>10}{'LiDAR avg':>12}{'LiDAR min':>12}")
    print(hdr); print("─" * W)
    specs = [
        ("Unscaled",     "unscaled_rmse","unscaled_abs_rel","unscaled_delta1",None,None),
        ("Median Scale", "median_rmse",  "median_abs_rel",  "median_delta1",  "median_scale",None),
        ("Mean Scale",   "mean_rmse",    "mean_abs_rel",    "mean_delta1",    "mean_scale",  None),
        ("Affine Scale", "affine_rmse",  "affine_abs_rel",  "affine_delta1",  "affine_scale","affine_shift"),
    ]
    for label, rc, ac, dc, sc, shc in specs:
        sv  = df[sc].mean()  if sc  else 1.0
        shv = df[shc].mean() if shc else 0.0
        print(
            f"  {label:<16}"
            f"{sv:>11.4f}{shv:>11.4f}"
            f"{df[rc].mean():>11.4f}{df[ac].mean():>11.4f}"
            f"{df[dc].mean():>10.4f}"
            f"{df['lidar_avg_depth'].mean():>12.4f}"
            f"{df['lidar_min_depth'].mean():>12.4f}"
        )
    print("═" * W)
    print(f"\n  Super-categories : {sorted(df['super_cat'].dropna().unique())}")
    print(f"  Sub-categories   : {sorted(df['sub_cat'].dropna().unique())}\n")

print("print_summary() defined.")


print_summary() defined.


## 13 · Run Inference  *(skip if CSV already exists)*

Set `SKIP_INFERENCE = True` to go straight to analysis using an existing CSV.


In [14]:
SKIP_INFERENCE = False   # ← set True to reload an existing CSV

if not SKIP_INFERENCE:
    log.info("── STEP 1/3: Indexing KITTI Eigen Split …")
    val_list = index_kitti(KITTI_ROOT, USE_TRAIN)

    log.info("── STEP 2/3: Loading UniDepth V2 ViTL14 …")
    unidepth_model = load_unidepth_v2(UNIDEPTH_WEIGHTS, CACHE_DIR, DEVICE)

    log.info("── STEP 3/3: Processing dataset (masks from %s) …", MASK_ROOT)
    results_df = run_evaluation()
else:
    log.info("Skipping inference — using existing CSV at %s", OUTPUT_CSV_PATH)
    results_df = load_results(OUTPUT_CSV_PATH)


12:43:43  INFO      ── STEP 1/3: Indexing KITTI Eigen Split …


Indexing Test:   0%|          | 0/652 [00:00<?, ?it/s]

12:43:44  INFO      Dataset indexed: 652 frames, 28 drives.
12:43:44  INFO      ── STEP 2/3: Loading UniDepth V2 ViTL14 …


ModuleNotFoundError: No module named 'unidepth.models'

## 14 · Analysis & Plots

In [ ]:
SKIP_PLOTS = False   # ← set True to only print summary

if os.path.exists(OUTPUT_CSV_PATH):
    log.info("Loading results CSV …")
    results_df = load_results(OUTPUT_CSV_PATH)
    print_summary(results_df)
    if not SKIP_PLOTS:
        generate_all_plots(results_df)
else:
    log.warning(
        "Results CSV not found at %s.  Run the inference cell first.",
        OUTPUT_CSV_PATH,
    )
